In [2]:
from IPython.display import display, HTML
display(HTML("""
<style>
div.container{width:86% !important;}
div.cell.code_cell.rendered{width:100%;}
div.CodeMirror {font-family:Consolas; font-size:12pt;}
div.output {font-size:12pt; font-weight:bold;}
div.input {font-family:Consolas; font-size:12pt;}
div.prompt {min-width:70px;}
div#toc-wrapper{padding-top:120px;}
div.text_cell_render ul li{font-size:12pt;padding:5px;}
table.dataframe{font-size:12px;}
</style>
"""))

In [37]:
# =============================
# 0) CONFIG
# =============================
from pathlib import Path

INPUT_CSV = Path("segment/06_1_chat_Persona_info_with_name.csv")
OUT_JSONL = Path("segment/07_chat_Persona_attributes_weighted.jsonl")
OUT_PREVIEW = Path("segment/07_chat_Persona_attributes_weighted_preview.json")

# 가중치 배분 비율(합=1 권장)
DEMO_SHARE = 0.30
BEHAV_SHARE = 0.70

# (선택) 인구통계 컬럼 직접 지정 (지정 시 자동탐지 무시)
DEMO_COLS_OVERRIDE = []
# 예시 DEMO_COLS_OVERRIDE = ["성별", "연령", "소득", "직업", "지역", "가족구성"]

# 인구통계 내부 항목별 가중
DEMO_FIELD_WEIGHTS = {'gender' :0.05, 'age':0.05, 'household':0.05, 'marriage':0.05, 'income_status':0.05, 'income_month':0.05,
                     "name":0, "job":0, "region":0, "education":0}
# 예시 DEMO_FIELD_WEIGHTS = { "연령": 2.0, "성별": 1.0, "소득": 1.0 }

# 성향 내부 multiplier (키는 *_scaled)
BEHAV_FIELD_MULTIPLIERS = {}
# 예시 BEHAV_FIELD_MULTIPLIERS = { "brand_loyalty_scaled": 1.2, "price_sensitivity_scaled": 0.9 }

# meta 필드로 고려할 칼럼명(있으면 자동으로 meta에 포함)
META_KEYS = ["cluster","label","Description"]

print("CONFIG loaded.")

CONFIG loaded.


In [38]:

# =============================
# 1) Load & clean
# =============================
import pandas as pd, re, numpy as np

def read_csv_fallback(path: Path):
    try:
        return pd.read_csv(path)
    except UnicodeDecodeError:
        return pd.read_csv(path, encoding="cp949")

df = read_csv_fallback(INPUT_CSV)

def clean_colname(c: str) -> str:
    return re.sub(r"'([^']+)'", r"\1", c)

df.rename(columns={c: clean_colname(c) for c in df.columns}, inplace=True)

print("Rows:", len(df))
print("Columns:", len(df.columns))
df.head(3)

Rows: 2124
Columns: 21


,id,name,gender,age,job,education,region,household,marriage,income_status,...,cluster,brand_loyalty_scaled,cooking_convenience_scaled,health_orientation_scaled,hmr_preference_scaled,premium_orientation_scaled,price_sensitivity_scaled,variety_seeking_scaled,label,description
0,1,한도윤,여자,50대,단순노무 종사자,고졸(대학 재학 포함),부산광역시,1인 가구,미혼(사별/이혼 포함),맞벌이 하지 않음,...,3,0.539552,0.597896,0.370502,0.804114,0.468730,0.315863,0.408696,트렌드 주도형 소비자,자신의 취향과 경험을 중시하는 소비자 그룹입니다. 단순히 배를 채우는 것 이상의 가...
1,2,강시우,남자,30대,기능원 및 관련 기능 종사자,고졸(대학 재학 포함),울산광역시,1인 가구,미혼(사별/이혼 포함),맞벌이 하지 않음,...,3,0.881343,0.662909,0.501363,0.902057,0.649511,0.350920,0.566522,트렌드 주도형 소비자,자신의 취향과 경험을 중시하는 소비자 그룹입니다. 단순히 배를 채우는 것 이상의 가...
2,3,이민준,남자,30대,판매 종사자,대학교 졸업(전문대졸/대학원생 포함),울산광역시,2세대가족,기혼,맞벌이 함,...,3,0.874627,0.680571,0.491003,0.921645,0.614984,0.334259,0.530435,트렌드 주도형 소비자,자신의 취향과 경험을 중시하는 소비자 그룹입니다. 단순히 배를 채우는 것 이상의 가...


In [39]:

# =============================
# 2) Auto-detect demographics & behavioral columns
# =============================
# behavior: *_scaled
behav_cols = [c for c in df.columns if c.endswith("_scaled")]

# demographics auto-detect
blacklist = {"id", "cluster","segment_id","label","description","Unnamed: 0"}
cat_candidates = []
for c in df.columns:
    if c in blacklist or c.endswith("_scaled"):
        continue
    if df[c].dtype == "object":
        nun = df[c].nunique(dropna=False)
        ratio = nun / max(1, len(df))
        if ratio <= 0.5:
            cat_candidates.append(c)

# numeric small-cardinality also considered
for c in df.columns:
    if c in blacklist or c.endswith("_scaled"):
        continue
    if np.issubdtype(df[c].dtype, np.number):
        nun = df[c].nunique(dropna=False)
        if 2 <= nun <= 20:
            cat_candidates.append(c)

if isinstance(DEMO_COLS_OVERRIDE, (list, tuple)) and len(DEMO_COLS_OVERRIDE) > 0:
    demo_cols = [c for c in DEMO_COLS_OVERRIDE if c in df.columns]
else:
    seen = set(); demo_cols = []
    for c in cat_candidates:
        if c not in seen:
            seen.add(c); demo_cols.append(c)
        if len(demo_cols) >= 9: break

if not behav_cols:
    raise RuntimeError("No behavioral (_scaled) columns detected.")

id_col = "id" if "id" in df.columns else None
print("Demographics:", demo_cols)
print("Behavioral:", behav_cols)
print("ID col:", id_col)

Demographics: ['name', 'gender', 'age', 'job', 'education', 'region', 'household', 'marriage', 'income_status']
Behavioral: ['brand_loyalty_scaled', 'cooking_convenience_scaled', 'health_orientation_scaled', 'hmr_preference_scaled', 'premium_orientation_scaled', 'price_sensitivity_scaled', 'variety_seeking_scaled']
ID col: id


In [56]:

# =============================
# 3) Helpers
# =============================
from typing import Dict, Any

def normalize_weights(raw: Dict[str, float], target_sum: float) -> Dict[str, float]:
    total = sum(v for v in raw.values() if pd.notna(v))
    if total <= 0:
        n = len(raw)
        return {k: (target_sum / n if n else 0.0) for k in raw}
    return {k: round((v / total) * target_sum, 3) for k, v in raw.items()}

def build_persona_attributes(row: pd.Series,
                             demo_cols, behav_cols,
                             demo_share: float, behav_share: float,
                             demo_field_weights: Dict[str, float],
                             behav_field_multipliers: Dict[str, float]) -> Dict[str, Any]:
    demo_raw = {c: demo_field_weights.get(c, 1.0) for c in demo_cols}
    demo_weights = normalize_weights(demo_raw, demo_share)
    demo_attrs = {c: {"value": (None if pd.isna(row.get(c)) else row.get(c)),
                      "weight": round(float(demo_weights.get(c, 0.0)), 3)}
                  for c in demo_cols}

    behav_raw = {}
    for c in behav_cols:
        val = row.get(c); 
        if pd.isna(val): val = 0.0
        mult = behav_field_multipliers.get(c, 1.0)
        behav_raw[c] = float(val) * float(mult)
    behav_weights = normalize_weights(behav_raw, behav_share)
    behav_attrs = {c: {"value": float(row.get(c, 0.0)),
                       "weight": float(behav_weights.get(c, 0.0))}
                   for c in behav_cols}

    attributes = {**demo_attrs, **behav_attrs}
    total_weight = sum(v["weight"] for v in attributes.values())
    if total_weight > 0:
        for k in attributes:
            attributes[k]["weight"] = attributes[k]["weight"] / total_weight
    return attributes

In [59]:

# =============================
# 4) Build & save with meta
# =============================
import json

total_share = DEMO_SHARE + BEHAV_SHARE
if abs(total_share - 1.0) > 1e-8:
    print(f"[WARN] DEMO_SHARE + BEHAV_SHARE = {total_share:.4f}. 자동 정규화합니다.")
    DEMO_SHARE = round(DEMO_SHARE / total_share, 3)
    BEHAV_SHARE = round(BEHAV_SHARE / total_share, 3)
    print(f" -> DEMO_SHARE={DEMO_SHARE:.3f}, BEHAV_SHARE={BEHAV_SHARE:.3f}")

records = []
for idx, row in df.iterrows():
    key = row.get("id", f"row_{idx}")
    # name = row.get("name", "미상")
    attrs = build_persona_attributes(row, demo_cols, behav_cols,
                                     DEMO_SHARE, BEHAV_SHARE,
                                     DEMO_FIELD_WEIGHTS, BEHAV_FIELD_MULTIPLIERS)
    meta = {}
    for k in META_KEYS:
        if k in df.columns:
            v = row.get(k)
            if pd.isna(v):
                v = None
            meta[k] = v
    
    for k, v in attrs.items():
        v["weight"] = round(v["weight"], 3)
        if isinstance(v.get('value'), float):
            v["value"] = round(v["value"], 3)

    rec = {"persona_key": key, # "persona_name" : name, 
           "attributes": attrs, "meta": meta}
    records.append(rec)

with OUT_JSONL.open("w", encoding="utf-8") as f:
    for r in records:
        f.write(json.dumps(r, ensure_ascii=False) + "\n")

print(f"Saved -> {OUT_JSONL} (records={len(records)})")

# preview
with OUT_PREVIEW.open("w", encoding="utf-8") as f:
    f.write(json.dumps(records[:3], ensure_ascii=False, indent=2))
print(f"Saved preview -> {OUT_PREVIEW}")
records[:1]

Saved -> segment\07_chat_Persona_attributes_weighted.jsonl (records=2124)
Saved preview -> segment\07_chat_Persona_attributes_weighted_preview.json


[{'persona_key': 1,
  'attributes': {'name': {'value': '한도윤', 'weight': 0.0},
   'gender': {'value': '여자', 'weight': 0.06},
   'age': {'value': '50대', 'weight': 0.06},
   'job': {'value': '단순노무 종사자', 'weight': 0.0},
   'education': {'value': '고졸(대학 재학 포함)', 'weight': 0.0},
   'region': {'value': '부산광역시', 'weight': 0.0},
   'household': {'value': '1인 가구', 'weight': 0.06},
   'marriage': {'value': '미혼(사별/이혼 포함)', 'weight': 0.06},
   'income_status': {'value': '맞벌이 하지 않음', 'weight': 0.06},
   'brand_loyalty_scaled': {'value': 0.54, 'weight': 0.108},
   'cooking_convenience_scaled': {'value': 0.598, 'weight': 0.119},
   'health_orientation_scaled': {'value': 0.371, 'weight': 0.074},
   'hmr_preference_scaled': {'value': 0.804, 'weight': 0.161},
   'premium_orientation_scaled': {'value': 0.469, 'weight': 0.094},
   'price_sensitivity_scaled': {'value': 0.316, 'weight': 0.063},
   'variety_seeking_scaled': {'value': 0.409, 'weight': 0.082}},
  'meta': {'cluster': 3, 'label': '트렌드 주도형 소비자'}}]